# Catalog figures from `results/comspec/gas_fit.csv`

Figures of the production-rate table (one row per comet and phase; `Q_<species>_status` is
`detected` ≥ 3σ, `marginal` 1–3σ, `upper_limit` / `negative_fit` with a 3σ limit, `not_covered`,
or `rejected`).  Every path comes from `spherex_comspec.directory` (`COMSPEC_ROOT`,
`COMSPEC_RESULT_DIR`, `COMSPEC_FIG_DIR` override it) and the style from `notebooks/rcparams.py`,
so the notebook runs from any kernel directory.  All figures are written to `fig/comspec/`.

| figure | content |
|---|---|
| `Q_vs_rh.png` | Q(H₂O), Q(CO₂), Q(CO) against ⟨r_h⟩ (log): detections filled, marginal values hollow, 3σ upper limits grey |
| `Q_vs_rh_multi_epoch.png` | Q(H₂O), Q(CO₂) of the comets with a value at ≥ `MIN_EPOCHS` phases; each comet's phases joined by a dotted line and labelled with its designation |
| `Q_over_afrho_vs_rh.png` | Q / A(0°)fρ against ⟨r_h⟩ at ρ = 10 000 and 20 000 km, for the phases that have both a gas value and a ZTF Afρ |


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# The project root is the directory that holds the packages, wherever the kernel started.
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "spherex_comspec" / "directory.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from spherex_comspec import directory as _dir
from spherex_comspec.config import SPECIES
from spherex_comspec.plotting import DPI_ONE, PRETTY, SP_COLORS, apply_rcparams

apply_rcparams()                       # notebooks/rcparams.py -- the 20-pt project style

GAS_FIT = _dir.RESULT_DIR / "gas_fit.csv"
FIG_DIR = _dir.FIG_DIR
ORBIT_CLASSES = _dir.ORBIT_CLASSES_CSV  # target slug -> SBDB designation, for the labels
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- selections ----------------------------------------------------------------------------
VALUE_STATUSES = ("detected", "marginal")            # rows that carry a value Q ± err
LIMIT_STATUSES = ("upper_limit", "negative_fit")     # rows that carry a 3σ upper limit
MIN_EPOCHS = 2                                       # phases with a value a comet needs in figure 2
AFRHO_METHODS = ("direct", "trend", "trend_extrap")  # how a ZTF Afρ was obtained ("none": no value)
APERTURES = (("10k", 10_000), ("20k", 20_000))       # the ρ of the attached afrho_<tag>_* columns
METHOD_MARKERS = {"direct": "o", "trend": "s", "trend_extrap": "^"}

print(f"root     {ROOT}\ngas fit  {GAS_FIT}\nfigures  {FIG_DIR}")


In [ ]:
df = pd.read_csv(GAS_FIT, dtype={"target": str})
df["phase"] = df["phase"].astype(int)

_names = pd.read_csv(ORBIT_CLASSES, dtype=str).drop_duplicates("key").set_index("key")["sbdb_des"]


def designation(slug: str) -> str:
    """SBDB designation of a target slug (``2023A3 -> C/2023 A3``); the slug itself when unknown."""
    return _names.get(slug, slug)


census = pd.DataFrame({s: df[f"Q_{s}_status"].value_counts() for s in SPECIES}).fillna(0).astype(int)
print(f"{len(df)} fits over {df.target.nunique()} comets")
census


In [ ]:
def rows(sp: str, status) -> pd.DataFrame:
    """The fits whose ``Q_<sp>_status`` is *status* (one string or a tuple of them)."""
    st = (status,) if isinstance(status, str) else tuple(status)
    return df[df[f"Q_{sp}_status"].isin(st)]


def style_rh_axis(ax):
    """log r_h with plain tick labels (1, 2, 3, 5, 10 au) instead of 2×10⁰."""
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator([1, 1.5, 2, 3, 5, 7, 10]))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(r"$\langle r_h \rangle$ [au]")
    ax.grid(alpha=0.3, which="both")


def plot_values(ax, d, sp, y, yerr, color, marker="o", ms=11, **kw):
    """Detected filled, marginal hollow -- the one convention every figure here uses."""
    det = (d[f"Q_{sp}_status"] == "detected").to_numpy()
    mar = (d[f"Q_{sp}_status"] == "marginal").to_numpy()
    x, y, yerr = d.r_hel_mean.to_numpy(), np.asarray(y, float), np.asarray(yerr, float)
    if det.any():
        ax.errorbar(x[det], y[det], yerr=yerr[det], fmt=marker, ms=ms, lw=1.5, capsize=4,
                    color=color, **kw)
    if mar.any():
        ax.errorbar(x[mar], y[mar], yerr=yerr[mar], fmt=marker, ms=ms, lw=1.2, capsize=3,
                    color=color, mfc="none", mew=2, **kw)


def value_handles(color="k", marker="o"):
    return [Line2D([], [], color=color, marker=marker, ls="none", ms=11, label="detected (≥ 3σ)"),
            Line2D([], [], color=color, marker=marker, ls="none", ms=11, mfc="none", mew=2,
                   label="marginal (1–3σ)")]


def save(fig, name: str) -> Path:
    path = FIG_DIR / name
    fig.savefig(path, dpi=DPI_ONE, bbox_inches="tight")
    print(f"-> {path}")
    return path


In [ ]:
# Figure 1: Q vs r_h per species -- detected, marginal, and the 3σ upper limits in grey
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, sp in zip(axes, SPECIES):
    val, lim = rows(sp, VALUE_STATUSES), rows(sp, LIMIT_STATUSES)
    ul = lim[f"Q_{sp}_upper_limit"]
    ax.errorbar(lim.r_hel_mean, ul, yerr=0.35 * ul, uplims=True, fmt="o", ms=7, lw=1.0,
                color="0.6", alpha=0.8, zorder=1)
    plot_values(ax, val, sp, val[f"Q_{sp}"], val[f"Q_{sp}_err"], SP_COLORS[sp], zorder=3)
    n_det = int((val[f"Q_{sp}_status"] == "detected").sum())
    n_mar = int((val[f"Q_{sp}_status"] == "marginal").sum())
    ax.set_title(f"Q({PRETTY[sp]})\n{n_det} detected,  {n_mar} marginal,  {len(lim)} limits", pad=12)
    ax.set_yscale("log")
    style_rh_axis(ax)
axes[0].set_ylabel(r"$Q$ [molecules s$^{-1}$]")
axes[0].legend(handles=value_handles() + [Line2D([], [], color="0.6", marker="v", ls="none", ms=9,
                                                 label="3σ upper limit")], loc="lower right")
fig.tight_layout()
save(fig, "Q_vs_rh.png")
plt.show()


In [ ]:
# Figure 2: comets with a value at >= MIN_EPOCHS phases -- the phases of one comet joined and labelled
def label_tracks(ax, anchors, fontsize=14, pad_px=8, gap_frac=0.034):
    """
    One label per comet to the right of its outermost point.

    Labels whose horizontal extents overlap are pushed apart vertically (bottom-up, in
    display space) so that they stay legible; a label that had to move keeps a thin leader
    to its point.  Text widths are estimated from the character count -- enough for these
    short designations.  *anchors*: (x, y, text, colour) in data coordinates.  Call it after
    ``tight_layout``: the axes geometry must be final.
    """
    fig = ax.figure
    fig.canvas.draw()
    px_per_pt, gap = fig.dpi / 72, ax.bbox.height * gap_frac
    items = []
    for x, y, text, color in anchors:
        xd, yd = ax.transData.transform((x, y))
        x0 = xd + pad_px
        items.append([x0, x0 + 0.62 * fontsize * px_per_pt * len(text), yd, yd, text, color, (x, y)])
    items.sort(key=lambda it: it[2])
    placed = []
    for it in items:
        for _ in range(len(items)):                  # bounded fixed-point iteration
            moved = False
            for p in placed:
                if it[0] < p[1] and it[1] > p[0] and abs(it[3] - p[3]) < gap:
                    it[3], moved = p[3] + gap, True
            if not moved:
                break
        placed.append(it)
    inv = ax.transData.inverted()
    for x0, _, yd, yl, text, color, xy in placed:
        arrow = dict(arrowstyle="-", color=color, lw=0.8, alpha=0.7) if abs(yl - yd) > 1 else None
        ax.annotate(text, xy=xy, xytext=inv.transform((x0, yl)), textcoords="data",
                    fontsize=fontsize, color=color, ha="left", va="center",
                    arrowprops=arrow, annotation_clip=False)


palette = plt.get_cmap("tab20").colors
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
anchors = {}                                         # species -> the label anchors of its panel
for ax, sp in zip(axes, ("H2O", "CO2")):
    val = rows(sp, VALUE_STATUSES)
    n_per = val.groupby("target").size()
    targets = sorted(n_per[n_per >= MIN_EPOCHS].index)
    anchors[sp] = []
    for i, t in enumerate(targets):
        d = val[val.target == t].sort_values("r_hel_mean")
        c = palette[i % len(palette)]
        ax.plot(d.r_hel_mean, d[f"Q_{sp}"], ls=":", lw=2, color=c, zorder=2)
        plot_values(ax, d, sp, d[f"Q_{sp}"], d[f"Q_{sp}_err"], c, zorder=3)
        last = d.iloc[-1]
        anchors[sp].append((last.r_hel_mean, last[f"Q_{sp}"], designation(t), c))
    ax.set_yscale("log")
    style_rh_axis(ax)
    lo, hi = ax.get_xlim()
    ax.set_xlim(lo, hi * 1.4)                        # room for the labels
    ax.set_title(f"Q({PRETTY[sp]}):  {len(targets)} comets with ≥ {MIN_EPOCHS} phases", pad=12)
    ax.legend(handles=value_handles(), loc="lower right")
axes[0].set_ylabel(r"$Q$ [molecules s$^{-1}$]")
fig.tight_layout()                                  # the labels need the final axes geometry
for ax, sp in zip(axes, ("H2O", "CO2")):
    label_tracks(ax, anchors[sp])
save(fig, "Q_vs_rh_multi_epoch.png")
plt.show()


In [ ]:
# Figure 3: Q / A(0°)frho vs r_h at rho = 10 000 and 20 000 km -- gas value and ZTF Afrho both present
fig, axes = plt.subplots(2, 3, figsize=(24, 15))
for (tag, rho), row_axes in zip(APERTURES, axes):
    for ax, sp in zip(row_axes, SPECIES):
        val = rows(sp, VALUE_STATUSES)
        a, a_err, method = val[f"afrho_{tag}_cm"], val[f"afrho_{tag}_err_cm"], val[f"afrho_{tag}_method"]
        d = val[a.notna() & (a > 0) & method.isin(AFRHO_METHODS)].copy()
        q, q_err = d[f"Q_{sp}"], d[f"Q_{sp}_err"]
        d["ratio"] = q / d[f"afrho_{tag}_cm"]
        d["ratio_err"] = d["ratio"] * np.hypot(q_err / q, d[f"afrho_{tag}_err_cm"] / d[f"afrho_{tag}_cm"])
        for m, marker in METHOD_MARKERS.items():
            dm = d[d[f"afrho_{tag}_method"] == m]
            plot_values(ax, dm, sp, dm.ratio, dm.ratio_err, SP_COLORS[sp], marker=marker, zorder=3)
        n_det = int((d[f"Q_{sp}_status"] == "detected").sum())
        n_mar = int((d[f"Q_{sp}_status"] == "marginal").sum())
        rho_km = f"{rho:,}".replace(",", " ")
        ax.set_title(f"Q({PRETTY[sp]}) / A(0°)fρ at ρ = {rho_km} km\n{n_det} detected,  {n_mar} marginal", pad=12)
        ax.set_yscale("log")
        style_rh_axis(ax)
    row_axes[0].set_ylabel(r"$Q$ / $A(0°)f\rho$  [molecules s$^{-1}$ cm$^{-1}$]")
METHOD_LABELS = {"direct": "Afρ from ZTF frames in the window", "trend": "Afρ from the r$_h$ law",
                 "trend_extrap": "Afρ from the law, extrapolated"}
method_handles = [Line2D([], [], color="k", marker=mk, ls="none", ms=11, mfc="none", mew=2,
                         label=METHOD_LABELS[m]) for m, mk in METHOD_MARKERS.items()]
fig.tight_layout()
fig.legend(handles=value_handles() + method_handles, loc="lower center", ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.035))
save(fig, "Q_over_afrho_vs_rh.png")
plt.show()


In [ ]:
for p in sorted(FIG_DIR.glob("Q_*.png")):
    print(f"{p.stat().st_size / 1e6:5.2f} MB  {p}")
